<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-09-multimodal-and-pretrained/lesson-9.2-gen-media/notebooks/GCP_Capstone_9.2_GenMedia.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 9.2 Generative Media — The Studio's Route, Chirp 3 HD From the List, and the Record Read Back
**Netsetos GenAI Engineering — GCP Capstone** · Module 9 · rebuilt on the live lane, 9 September 2026

Image generation and speech, on the lane. The first image is generated through `/v1/media/generate` - the Studio's server half, deployed with the API - which checks the roster, spends once, caches the asset and writes the audit row; only then does the notebook make the same call with the SDK to walk the interleaved parts. Voices come from `list_voices`, not from a hardcoded name. Chirp 3 HD reads a cited answer aloud. The outsider is refused before any spend, and the audit row is read back from the retention-locked bucket.


## Setup


In [ ]:
!pip install -q google-genai==2.22.0 google-cloud-texttospeech==2.37.0 google-cloud-storage==3.13.1 Pillow==12.3.0 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp-learners"   # the kit: deploy/shared is the tool layer every lesson on the lane imports
BRANCH     = "main"        # the learner repo's branch: the notebooks and the kit (deploy/) ship there together

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google import genai
from google.genai import types

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
API_URL       = f"https://documind-api-{NUMBER}.{REGION}.run.app"
UPLOAD_BUCKET = f"{PROJECT_ID}-uploads"     # storage.tf: the bucket eventarc.tf watches - the corpus, media included
MEDIA_BUCKET  = f"{PROJECT_ID}-media"       # storage.tf: generated assets, 30-day lifecycle (a cache, not a record)
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": API_URL,
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
MEMBER_SA   = os.environ["DOCUMIND_IMPERSONATE_SA"]
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"   # IAM admits it, no roster does (4.8, 7.2)

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - the contract gate fails a paste.
gen = genai.Client(enterprise=True, project=PROJECT_ID, location="global")   # every generate_content in this lesson

print("kit:", KIT, "| API:", API_URL, "| media:", f"gs://{MEDIA_BUCKET}")


## Cell 1: The corpus's media, and the API's routes
`api()` posts to the API as the roster member; `api_as()` posts as somebody else; `audit_rows()` lists the audit bucket.


In [ ]:
import json, requests, time
from google.cloud import storage

# THE ROUTES 9.4's Studio stands on, called the way the UI calls them: one ID token per request,
# minted AS the roster member, audience = the API (7.3's hour-long fuse never arms). The body names
# the tenant; the API checks the caller's email on that tenant's roster before it spends a paisa.
def api(path: str, body: dict | None = None, timeout: int = 120) -> tuple[int, dict | str]:
    """POST one API route as documind-ui-sa. Returns (status, json-or-text) - never raises on 4xx,
    because a refusal is data this module reads (the outsider cells)."""
    r = requests.post(f"{API_URL}{path}", json=body,
                      headers={"Authorization": f"Bearer {documind_tools._id_token(API_URL)}"}, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

# The corpus's media objects, read as YOU (the Colab credential is a project owner; the lane's
# services read them as their own accounts). One client, both buckets.
gcs = storage.Client(project=PROJECT_ID)

def media_objects(prefix: str = f"{TENANT}/") -> list[str]:
    """Every image, video or recording under the tenant's prefix of the uploads bucket."""
    return sorted(b.name for b in gcs.list_blobs(UPLOAD_BUCKET, prefix=prefix)
                  if b.name.lower().endswith((".png", ".jpg", ".jpeg", ".mp4", ".mp3")))

def gcs_bytes(uri: str) -> bytes:
    bucket, _, name = uri.removeprefix("gs://").partition("/")
    return gcs.bucket(bucket).blob(name).download_as_bytes()

FIG3  = f"gs://{UPLOAD_BUCKET}/{TENANT}/annual_report_2026_fig3.png"
INV   = f"gs://{UPLOAD_BUCKET}/{TENANT}/inv_2026_0412.png"
PAGE  = f"gs://{UPLOAD_BUCKET}/{TENANT}/payment_of_bonus_act_1965_p30.png"
VIDEO = f"gs://{UPLOAD_BUCKET}/{TENANT}/townhall_2026_q1.mp4"
POSH  = f"gs://{UPLOAD_BUCKET}/{TENANT}/posh_act_2013.pdf"

have = media_objects()
HAS_VIDEO = f"{TENANT}/townhall_2026_q1.mp4" in have
print("media in the corpus:", have or "NONE - run `make media` and `make ingest-corpus` from deploy/ (README: media is a document)")
print("video:", "present" if HAS_VIDEO else "absent (make media MEDIA_ARGS=--video, or drop a recording in) - the video cells will say so")
assert have, "no media under the tenant's prefix: nothing in this lesson can cite a figure until the corpus holds one"


In [ ]:
from google.auth import impersonated_credentials
from google.auth.transport.requests import Request

def id_token_as(service_account: str, audience: str) -> str:
    """A Google ID token minted AS a service account, for one audience, with the email (4.8, 7.3)."""
    source, _ = google.auth.default()
    target = impersonated_credentials.Credentials(source_credentials=source, target_principal=service_account,
                                                  target_scopes=["https://www.googleapis.com/auth/cloud-platform"])
    idc = impersonated_credentials.IDTokenCredentials(target, target_audience=audience, include_email=True)
    idc.refresh(Request())
    return idc.token

def api_as(service_account: str, path: str, body: dict) -> tuple[int, dict | str]:
    """The same route, as a DIFFERENT account - the outsider cells."""
    r = requests.post(f"{API_URL}{path}", json=body,
                      headers={"Authorization": f"Bearer {id_token_as(service_account, API_URL)}"}, timeout=120)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]


In [ ]:
import datetime

def audit_rows(action: str, tenant: str = TENANT, minutes: int = 30) -> list[dict]:
    """The audit trail, read back. shared/audit_log.py writes ONE JSON object per event into the
    retention-locked audit bucket, named {year}/{month}/{day}/{tenant}/{action}-{id}.json - so the
    record of who generated what is a listing, not a query. The media bucket is only a cache; this
    is the record."""
    now = datetime.datetime.now(datetime.timezone.utc)
    since = now - datetime.timedelta(minutes=minutes)
    rows = []
    for day in sorted({since.strftime("%Y/%m/%d"), now.strftime("%Y/%m/%d")}):
        for blob in gcs.list_blobs(f"{PROJECT_ID}-audit", prefix=f"{day}/{tenant}/{action}-"):
            if blob.updated and blob.updated < since:
                continue
            rows.append(json.loads(blob.download_as_text()))
    return sorted(rows, key=lambda r: r.get("ts", ""))


## Cell 2: Generate through the route
Roster first, then spend, then the audit and usage rows. The blob is named by the prompt's hash, which is the cache key; the second call is a cache hit under `DEMO_MODE=1`.


In [ ]:
import hashlib
from io import BytesIO
from PIL import Image
from IPython.display import display

# THE STUDIO'S ROUTE, BEFORE THE SDK. /v1/media/generate is 9.4's server half, deployed with the API
# (services/rag-api/media.py). It checks the caller on the tenant's ROSTER, spends once, stores the PNG
# as {tenant}/gen/<sha256 of the prompt>.png in the media bucket, writes the audit row (prompt hashed,
# never stored) and the usage row tenant_daily reads. Under DEMO_MODE=1 - the lean deploy - a repeated
# prompt is a cache hit: one bill per prompt, however many times the room re-runs the cell.
PROMPT = ("A clean, professional grouped bar chart titled 'ACME revenue by region, FY2025 vs FY2026 (Rs crore)': "
          "India 412 to 508, APAC 188 to 236, EMEA 96 to 91, Americas 143 to 170. Teal palette, value labels, white background.")
PROMPT_SHA = hashlib.sha256(PROMPT.encode()).hexdigest()[:32]

status, out = api("/v1/media/generate", {"prompt": PROMPT, "tenant_id": TENANT})
assert status == 200, (status, out)
print("blob   :", out["blob"], "| cached:", out["cached"])
assert out["blob"].endswith(f"{PROMPT_SHA}.png"), "the blob is named by the prompt's hash - that IS the cache key"

png = gcs_bytes(f"gs://{out.get('bucket', MEDIA_BUCKET)}/{out['blob']}")
img = Image.open(BytesIO(png))
print("image  :", img.size, img.mode, f"{len(png) // 1024} KB")
display(img)

status, again = api("/v1/media/generate", {"prompt": PROMPT, "tenant_id": TENANT})
print("again  : cached =", again.get("cached"), "(DEMO_MODE=1 on the lean deploy: the same prompt is the same asset, and costs nothing)")


## Cell 3: The SDK call the route makes
Text and image come back interleaved. Walk the parts.


In [ ]:
# THE SDK CALL THE ROUTE MAKES. TEXT and IMAGE come back INTERLEAVED in one parts list - the model
# narrates what it drew - so walk the parts; parts[0] is not your PNG. gemini-3.1-flash-image is the
# image model; reading an image (9.1's captions) is a text task and uses gemini-3.6-flash.
r = gen.models.generate_content(
    model="gemini-3.1-flash-image",
    contents="An infographic in a teal palette with three tiles: '85% cost reduction', '3x faster processing', "
             "'99.2% accuracy'. Title: DocuMind AI Results. Clean, white background, no small print.",
    config=types.GenerateContentConfig(response_modalities=["TEXT", "IMAGE"]))
image_part = None
for part in r.candidates[0].content.parts:
    if part.text:
        print("text :", part.text.strip()[:140])
    elif part.inline_data:
        image_part = Image.open(BytesIO(part.inline_data.data))
        image_part.save("infographic.png")
        print("image:", image_part.size, "-> infographic.png (SynthID watermark inside the pixels, always on)")
assert image_part is not None, "no image part came back - read the text part: the model declined or narrated instead"
display(image_part)


## Cell 4: Editing is a chat


In [ ]:
# EDITING IS A CHAT, NOT A NEW PROMPT. Re-prompting from scratch gets you a different chart; a chat
# turn edits the one you already have, because the previous image stays in the context window.
# Each turn carries every previous image, so a long edit chain costs more per turn than the one before.
chat = gen.chats.create(model="gemini-3.1-flash-image",
                        config=types.GenerateContentConfig(response_modalities=["TEXT", "IMAGE"]))
first = chat.send_message("Draw a bar chart of ACME revenue by region for FY2026: India 508, APAC 236, EMEA 91, Americas 170 (Rs crore).")
second = chat.send_message("Keep everything. Make the EMEA bar red and add the label '-5.2% vs FY2025' above it.")
for name, resp in (("first", first), ("edit", second)):
    im = next((Image.open(BytesIO(p.inline_data.data)) for p in resp.candidates[0].content.parts if p.inline_data), None)
    assert im is not None, f"{name}: no image part"
    im.save(f"chart_{name}.png"); print(name, im.size)
    display(im)


## Cell 5: Voices from the API
The Chirp 3 HD list moves. `list_voices`, filtered, and the choice made from what is actually there.


In [ ]:
from google.cloud import texttospeech as tts
tts_client = tts.TextToSpeechClient()

# VOICE NAMES COME FROM THE API, NEVER FROM A NOTEBOOK. The Chirp 3 HD list moves; a hardcoded name is
# a 400 in the room. list_voices, filtered, and the choice made from what is actually there.
voices = {v.name: list(v.language_codes) for v in tts_client.list_voices().voices if "Chirp3-HD" in v.name}
for lang in ("en-IN", "hi-IN", "ta-IN", "te-IN", "bn-IN"):
    names = sorted(n for n, langs in voices.items() if lang in langs)
    print(f"{lang}: {len(names):2} Chirp 3 HD voices   {names[:3]}{' ...' if len(names) > 3 else ''}")

EN_IN = sorted(n for n, langs in voices.items() if "en-IN" in langs)
HI_IN = sorted(n for n, langs in voices.items() if "hi-IN" in langs)
assert EN_IN and HI_IN, "no Chirp 3 HD voice listed for en-IN / hi-IN - is the Text-to-Speech API enabled? (make apis)"
VOICE_EN = next((n for n in EN_IN if n.endswith("Kore")), EN_IN[0])
VOICE_HI = next((n for n in HI_IN if n.endswith("Kore")), HI_IN[0])
print("\nusing", VOICE_EN, "and", VOICE_HI, "- the UI's voice.py defaults to en-IN-Chirp3-HD-Kore and falls back the same way")


## Cell 6: The lane's answer, read aloud
What Chirp reads is a cited fact (golden row lk-06). The UI's `voice.cached_tts` does the same behind the *Read answers aloud* toggle.


In [ ]:
from IPython.display import Audio

# READ THE LANE'S ANSWER ALOUD. The text is retrieve()'s - golden row lk-06, the notice period for a
# confirmed E3 - so what Chirp reads is a CITED fact, not a script. The UI does exactly this behind the
# "Read answers aloud" toggle (voice.cached_tts), cached in the TTS bucket by the SHA-256 of voice and
# text, so the second reading of the same answer is free.
ans = documind_tools.retrieve("What is the notice period for a confirmed E3?", tenant_id=TENANT, brain="direct")
text = (ans.get("answer") or "")[:600]
assert ans.get("answerable") and "60" in text.replace("sixty", "60"), text

def speak(text: str, voice: str, lang: str, encoding=tts.AudioEncoding.MP3) -> bytes:
    r = tts_client.synthesize_speech(input=tts.SynthesisInput(text=text),
                                     voice=tts.VoiceSelectionParams(language_code=lang, name=voice),
                                     audio_config=tts.AudioConfig(audio_encoding=encoding))
    return r.audio_content

mp3 = speak(text, VOICE_EN, "en-IN")
open("answer_en.mp3", "wb").write(mp3)
print(f"{len(mp3) // 1024} KB of MP3 for {len(text)} characters of a cited answer:")
print("  ", text[:160], "...")
assert len(mp3) > 8_000
Audio(mp3)


## Cell 7: Hindi


In [ ]:
# THE SAME SENTENCE IN HINDI. Chirp 3 HD has hi-IN, ta-IN, te-IN and more; the voice came from the list
# above. (Translating the lane's whole answer is 9.3's job - the Translation API on a real answer.)
hindi = "सूचना अवधि साठ दिन है। कृपया अपनी टीम के प्रबंधक से बात करें।"      # the notice period is sixty days
mp3_hi = speak(hindi, VOICE_HI, "hi-IN")
open("answer_hi.mp3", "wb").write(mp3_hi)
print(VOICE_HI, f"{len(mp3_hi) // 1024} KB")
Audio(mp3_hi)


## Cell 8: The outsider, at the Studio's door
Refused by the roster before the model is called - so the refusal costs nothing and generates nothing.


In [ ]:
# THE OUTSIDER, AT THE STUDIO'S DOOR. documind-outsider-sa is the eval gate's fixture account (4.8,
# 7.2): IAM lets it invoke the API, and no roster lists it. The route checks the roster BEFORE the
# model is called - the auth-wiring gate asserts that order in media.py - so the refusal costs nothing
# and generates nothing. Spending first and checking later is the bug this cell exists to make visible.
status, body = api_as(OUTSIDER_SA, "/v1/media/generate", {"prompt": PROMPT, "tenant_id": TENANT})
print("outsider ->", status, str(body)[:120])
assert status == 403, f"the outsider was not refused by the roster: {status} {body}"


## Cell 9: The record, read back
The media bucket is a cache; the audit bucket is the record. The row names the actor, the target and the prompt's hash.


In [ ]:
# THE RECORD, READ BACK. The media bucket is a cache (30 days). The audit bucket is the record: one JSON
# object per event, retention-locked for five years, written by the one shared writer. It names who
# generated what, under which tenant, with the prompt's HASH - never the prompt, because you cannot
# delete a mistake out of a locked bucket. Logs lag a little; the loop waits for the row.
for attempt in range(6):
    rows = [r for r in audit_rows("media.generate") if r.get("meta", {}).get("prompt_sha") == PROMPT_SHA]
    if rows:
        break
    time.sleep(5)
assert rows, "no media.generate row with this prompt's hash in the audit bucket yet - re-run in a minute"
row = rows[-1]
print("action :", row["action"], "| ts:", row["ts"])
print("actor  :", row["actor"])
print("target :", row["target"])
print("meta   :", row["meta"])
assert "prompt" not in row["meta"] and row["meta"]["synthid"] is True
assert row["actor"]["user_email"] == MEMBER_SA, "the actor is the identity the API verified - the roster member this notebook mints as"


## Cell 10: What it costs


In [ ]:
# WHAT GENERATIVE MEDIA COSTS, IN THE ROW'S OWN UNITS. media.py's usage row carries cost_usd=0.039 per
# image (0 on a cache hit) in the shape tenant_daily reads; TTS is per character with a free tier.
USD_INR = 85
IMAGE_USD, TTS_PER_M, TTS_FREE = 0.039, 30.0, 1_000_000          # verified 2026-09-04; re-verify on the pricing page
images_per_month, repeats = 500, 3                                 # 500 prompts, each re-run three times in demos
paid = images_per_month                                            # DEMO_MODE: the repeats are cache hits
tts_chars = 500 * 4000
tts_usd = max(0, tts_chars - TTS_FREE) * TTS_PER_M / 1_000_000
print(f"images   : {images_per_month} prompts x {repeats} runs = {images_per_month * repeats} calls, {paid} billed = ${paid * IMAGE_USD:.2f}")
print(f"          (without the cache: ${images_per_month * repeats * IMAGE_USD:.2f})")
print(f"TTS      : {tts_chars:,} chars, first {TTS_FREE:,} free = ${tts_usd:.2f}")
print(f"SynthID  : $0.00 - a property of the output, not a feature you buy")
total = paid * IMAGE_USD + tts_usd
print(f"TOTAL    : ${total:.2f} / month = Rs {total * USD_INR:,.0f}")


## Cell 11: SynthID is not an API you call


In [ ]:
# SynthID is NOT an API you call - it is automatic
# This cell documents what happens behind the scenes

synthid_coverage = {
    'Gemini image generation': 'Always on (pixel-level watermark)',
    'Chirp 3 HD audio': 'Always on (frequency-level watermark)',
    'Veo video': 'Always on (per-frame watermark)',
    'Gemini text (consumer app)': 'On in consumer app, not reliable via API',
}

print('SynthID Watermark Coverage:')
for service, status in synthid_coverage.items():
    print(f'  {service}: {status}')

print('\nDetection Methods:')
print('  1. SynthID Detector portal (waitlist)')
print('  2. Gemini App upload (~10 images/day, ~3h audio/day)')

print('\nCompliance:')
print('  EU AI Act Article 50 (effective Aug 2, 2026)')
print('  Requires machine-readable marking of AI content')
print('  Fine: up to EUR 15M or 3% global turnover')
print('  DocuMind: COMPLIANT (SynthID + C2PA automatic)')


## Where this goes
- **9.4** is the Studio end to end: the same route from a Streamlit tab, video segments, the signed upload that becomes an ingest.
- **9.3** puts Chirp 3 in the other direction: the answer this lesson spoke, transcribed back, with a word error rate.

## ✅ Lesson 9.2 complete
- ✅ An image generated through the lane's route: roster, one spend, a cached second call
- ✅ The interleaved parts walked; an edit as a chat turn
- ✅ Chirp 3 HD voices from `list_voices`, a cited answer read aloud, Hindi
- ✅ The outsider refused before any spend
- ✅ The audit row read back: actor, target, prompt hash, never the prompt
- ✅ Cost in the usage row's units; SynthID as a property of the output
